In [3]:
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, compare_to_recontours, save_evaluation_results
from uncertainty_util import remove_negative_prompts, remove_positive_prompts
from pathlib import Path
import numpy as np

root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

methods_available = ["raycast", "local_normals"]
propagation_styles = ['default', 'full', 'prompt_based', 'central_start', 'central_partitions']
method = methods_available[1]
propagation_style = propagation_styles[3]

rootpath = Path(root)
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()])
print(subjects)



['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']


In [ ]:
def pipeline(target_mm,method,propagation_style,unc_band_thr_mm,pixel_interval,angle_step,weighting_list,results_filename):
    nn_Unet_Evals = []
    SAM_Evals = []
    Recontour_evals = []
    for subject_nr in range(len(subjects)):
        data = DataLoader(parentfolder=root,subject_nr=subject_nr,volume_of_interest="CTVT",verbose=True)
        unc_handler = UG_prompter(data=data)
        seg_handler = Segmentation(data=data)

        unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=target_mm, method="raycast", mode="median") #unc_threshold=0.033470
        unc_handler.compute_band_thickness(method=method)

        nietjes_prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=unc_band_thr_mm,
        interpix_dist=3,
        pixel_interval=pixel_interval,
        angle_step=angle_step,
        method=method)

        bbox_prompts = unc_handler.generate_prompts_boxes(band_threshold=0.0)

        dense_prompt = seg_handler.load_dense_prompt()
        dense_and_nietjes_prompts = combine_prompt_sets(prompt_dict_list = [dense_prompt, nietjes_prompts])

        prompt_sets = [dense_and_nietjes_prompts, bbox_prompts]
        prompt_names = ["Dense_and_nietjes", "Uncertainty_bboxes"]
        
        prompt_weights = weighting_list

        seg_handler.compile_prompt_sets(prompt_dict_list=prompt_sets, prompt_set_names=prompt_names, prompt_set_weights=prompt_weights)

        seg_handler.run_segmentation_sets(propagation_style=propagation_style, weighting_strategy="custom", threshold=0.0)
        seg_handler.remove_distant_slices(tolerance_frames=0)

        eval_handler = Evaluator(segmentation=seg_handler)
        metrics = eval_handler.compute_all(surface_dice_tol=1.0)
        SAM_Evals.append(metrics)

        save_evaluation_results(SAM_Evals, "Dense_Bbox_weighting_experiment_results", filename=results_filename)

    return SAM_Evals, nn_Unet_Evals, Recontour_evals

In [4]:
def extract_logits_from_strats(output_folder):
    
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    for subject_nr in range(len(subjects)):
        data = DataLoader(parentfolder=root,subject_nr=subject_nr,volume_of_interest="CTVT",verbose=True)
        unc_handler = UG_prompter(data=data)
        seg_handler = Segmentation(data=data)

        unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=3.0, method="raycast", mode="median") #unc_threshold=0.033470
        unc_handler.compute_band_thickness(method="local_normals")

        #Dense + nietjes prompt strategy generation
        nietjes_prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=4.0,
                interpix_dist=5,
                pixel_interval=10,
                angle_step=5,
                method="local_normals")

        #Bbox prompt strategy generation
        bbox_prompts = unc_handler.generate_prompts_boxes(band_threshold=3.0)

        # indiscriminate_nietjes = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=0.0,
        #         interpix_dist=5,
        #         pixel_interval=25,
        #         angle_step=5,
        #         method="local_normals")
        # indiscriminate_positive = remove_negative_prompts(indiscriminate_nietjes)
        # nietjes_negatives = remove_positive_prompts(nietjes_prompts)
        # updated_staples = combine_prompt_sets([indiscriminate_positive, nietjes_negatives])

        dense_prompt = seg_handler.load_dense_prompt()
        dense_and_nietjes_prompts = combine_prompt_sets(prompt_dict_list = [dense_prompt, nietjes_prompts])

        prompt_sets = [dense_and_nietjes_prompts, bbox_prompts]
        prompt_names = ["Dense_and_nietjes", "Uncertainty_bboxes"]
        prompt_weights = [0.3, 0.7]#Isn't used here

        seg_handler.compile_prompt_sets(prompt_dict_list=prompt_sets, prompt_set_names=prompt_names, prompt_set_weights=prompt_weights)

        seg_handler.run_segmentation_sets(propagation_style="central_start", weighting_strategy="custom", threshold=0.0, no_external_propagation=True)

        #saving logits for each subject. Easier for weighting
        subject_name = data.subject_name
        savepath = output_folder / f"{subject_name}_logits.npz"

        np.savez_compressed(
            savepath,
            **seg_handler.logits_per_set,
        )

        print(f"Saved logits for {subject_name} to:\n{savepath}")


In [5]:
extract_logits_from_strats(output_folder=r"C:\Users\20202310\Desktop\saved_logits_new")

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


use_mask_input_as_output_without_sam: True
iter=00 | thr=0.194962 | band=0.94 mm | error=2.06
iter=01 | thr=0.097481 | band=1.41 mm | error=1.59
iter=02 | thr=0.048740 | band=1.99 mm | error=1.01
iter=03 | thr=0.024370 | band=2.46 mm | error=0.54
iter=04 | thr=0.012185 | band=2.93 mm | error=0.07
iter=05 | thr=0.006093 | band=3.63 mm | error=0.63
iter=06 | thr=0.009139 | band=3.28 mm | error=0.28
iter=07 | thr=0.010662 | band=3.16 mm | error=0.16
iter=08 | thr=0.011424 | band=3.05 mm | error=0.05
iter=09 | thr=0.011804 | band=3.05 mm | error=0.05
iter=10 | thr=0.011995 | band=3.05 mm | error=0.05
iter=11 | thr=0.012090 | band=2.93 mm | error=0.07
iter=12 | thr=0.012042 | band=2.93 mm | error=0.07
iter=13 | thr=0.012019 | band=2.99 mm | error=0.01
iter=14 | thr=0.012007 | band=2.99 mm | error=0.01
iter=15 | thr=0.012001 | band=2.99 mm | error=0.01
iter=16 | thr=0.011998 | band=3.05 mm | error=0.05
iter=17 | thr=0.011999 | band=2.99 mm | error=0.01
iter=18 | thr=0.011998 | band=2.99 mm |

c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (central_start, from slice 36)...


...

Running segmentation for prompt set 'Uncertainty_bboxes' with slices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Adding prompt(s) on slice 45
Adding prompt(s) on slice 46
Adding prompt(s) on slice

propagate in video: 100%|██████████| 10/10 [00:00<00:00, 26.25it/s]


Backward propagation (central_start, from slice 42)...


propagate in video: 100%|██████████| 10/10 [00:00<00:00, 44.63it/s]


Saved logits for newAcq_4a136e8fe320bd13 to:
C:\Users\20202310\Desktop\saved_logits_new\newAcq_4a136e8fe320bd13_logits.npz


In [ ]:
from pathlib import Path
output_folder = Path("Dense_Bbox_weighting_experiment_results")
output_folder.mkdir(parents=True, exist_ok=True)

parameter_log_path = output_folder / "run_parameters.txt"

weighting_lists = [[0.0,1.0],[0.1,0.9],[0.2,0.8],[0.3,0.7],[0.4,0.6],[0.5,0.5],[0.6,0.4],[0.7,0.3],[0.8,0.2],[0.9,0.1],[1.0,0.0]]
run_nr = 1

with open(parameter_log_path, "w") as f:
    f.write("Run parameter overview\n")
    f.write("=" * 80 + "\n\n")

    for weighting_list in weighting_lists:

        results_filename = output_folder / f"run{run_nr}.csv"

        f.write(f"run{run_nr}.csv\n")
        f.write(f"  target_mm: 3.0\n")
        f.write(f"  method: local_normals\n")
        f.write(f"  unc_band_thr_mm: 4.0\n")
        f.write(f"  pixel_interval: 10\n")
        f.write(f"  weighting_list: {weighting_list}\n")
        f.write("\n")


        pipeline(
            target_mm=3.0,
            method="local_normals",
            propagation_style="central_start",
            unc_band_thr_mm=4.0,
            pixel_interval=10,
            angle_step=1.0,
            weighting_list=weighting_list,
            results_filename=f"Staple{weighting_list[0]}_bbox{weighting_list[1]}",
        )

        run_nr += 1